# RAG Prompt and Retrieval Optimization

This notebook evaluates a Wikipedia-based retrieval-augmented generation pipeline using Pyserini/Lucene and Llama 3.1 8B Instruct. It compares prompt strategies and retrieval depths on 200 HotpotQA development examples using Exact Match and F1.

In [ ]:
import os
import json
import re
import sys
import string
from collections import Counter

import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from huggingface_hub import login
from pyserini.search.lucene import LuceneSearcher
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Python executable:", sys.executable)

In [ ]:
dev_path = "data/dev.jsonl"

dev_data = []
with open(dev_path) as f:
    for line in f:
        dev_data.append(json.loads(line))

print("Dev size:", len(dev_data))
print(dev_data[0])

In [ ]:
subset_size = 200
subset = dev_data[:subset_size]

questions = [ex["question"] for ex in subset]
answers   = [ex["answer"]   for ex in subset]

print("Loaded", len(questions), "questions")

In [ ]:
import os

WIKI_INDEX_PATH = os.environ.get(
    "WIKI_INDEX_PATH",
    "/home/ktamiras/wiki_index"
)

searcher = LuceneSearcher(WIKI_INDEX_PATH)
print("Searcher loaded!")

In [ ]:
login()

model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded on:", next(model.parameters()).device)

In [ ]:
def retrieve_context(question, k=4):
    hits = searcher.search(question, k)
    passages = []
    for hit in hits:
        doc = searcher.doc(hit.docid)
        raw = doc.raw()
        try:
            parsed = json.loads(raw)
            text = parsed.get("contents", raw)
        except Exception:
            text = raw
        passages.append(text)
    return "\n\n".join(passages)

def build_contexts(questions, k=4):
    contexts = []
    for i, q in enumerate(questions):
        if i % 20 == 0:
            print(f"Retrieving {i}/{len(questions)} with k={k}")
        contexts.append(retrieve_context(q, k=k))
    print(f"Done building {len(contexts)} contexts for k={k}")
    return contexts

In [ ]:
def build_prompt(question, context, mode="baseline"):
    if mode == "baseline":
        instruction = (
            "Answer the question using the provided context."
        )

    elif mode == "short":
        instruction = (
            "Answer the question using the provided context. "
            "Respond with ONLY the final answer phrase. "
            "Do not include explanations or extra words."
        )

    elif mode == "exact_span":
        instruction = (
            "Answer the question using the provided context. "
            "Copy ONLY the exact answer span from the context. "
            "Do not write a full sentence. "
            "Do not include explanation."
        )

    elif mode == "best_guess":
        instruction = (
            "Answer the question using the provided context. "
            "Give your best short answer even if the context is incomplete. "
            "Do not say you cannot find the answer. "
            "Respond with only the final answer phrase."
        )

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return f"""Context:
{context}

Question: {question}

{instruction}

Answer:"""

In [ ]:
device = next(model.parameters()).device
def generate_batch(prompts, batch_size=1, max_input_len=512, max_new_tokens=20):
    """
    Generates answers for a list of prompts using your loaded model + tokenizer.
    """
    
    outputs_all = []
    model.eval()
    
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_len
        ).to(device)
        
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Only keep generated tokens
        gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        
        outputs_all.extend(decoded)
        
        del inputs, out_ids, gen_ids
        torch.cuda.empty_cache()
    
    return outputs_all

In [ ]:
def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    
    def white_space_fix(text):
        return ' '.join(text.split())
    
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    
    def lower(text):
        return text.lower()
    
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    if num_same == 0:
        return 0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def evaluate_predictions(predictions, references):
    em = 0
    f1 = 0
    
    for pred, ref in zip(predictions, references):
        em += exact_match_score(pred, ref)
        f1 += f1_score(pred, ref)
    
    total = len(predictions)
    return {
        "EM": em / total,
        "F1": f1 / total
    }

In [ ]:
def clean_prediction(pred):
    pred = pred.strip()

    # keep only first line
    pred = pred.split("\n")[0].strip()

    # remove leading "Answer:" if present
    if pred.lower().startswith("answer:"):
        pred = pred[len("answer:"):].strip()

    # remove outer quotes
    pred = pred.strip('"').strip("'").strip()

    # optionally cut at first sentence-ending punctuation if output is long
    pred = re.split(r"[.?!]", pred)[0].strip()

    return pred

In [ ]:
def run_prompt_experiment(
    name,
    questions,
    answers,
    contexts,
    prompt_mode,
    apply_cleaning=True,
    save_dir="final_results"
):
    os.makedirs(save_dir, exist_ok=True)

    prompts = [
        build_prompt(q, ctx, mode=prompt_mode)
        for q, ctx in zip(questions, contexts)
    ]

    preds = generate_batch(prompts)

    if apply_cleaning:
        preds = [clean_prediction(p) for p in preds]

    metrics = evaluate_predictions(preds, answers)

    # save predictions
    pred_path = os.path.join(save_dir, f"{name}_predictions.jsonl")
    with open(pred_path, "w") as f:
        for q, a, p in zip(questions, answers, preds):
            f.write(json.dumps({
                "question": q,
                "gold": a,
                "prediction": p
            }) + "\n")

    print(f"{name}: EM={metrics['EM']:.6f}, F1={metrics['F1']:.6f}")
    return {
        "name": name,
        "prompt_mode": prompt_mode,
        "EM": metrics["EM"],
        "F1": metrics["F1"],
        "predictions": preds
    }

In [ ]:
contexts_k4 = build_contexts(questions, k=4)

In [ ]:
results = []

results.append(
    run_prompt_experiment(
        name="baseline_prompt",
        questions=questions,
        answers=answers,
        contexts=contexts_k4,
        prompt_mode="baseline"
    )
)

results.append(
    run_prompt_experiment(
        name="short_answer_prompt",
        questions=questions,
        answers=answers,
        contexts=contexts_k4,
        prompt_mode="short"
    )
)

results.append(
    run_prompt_experiment(
        name="exact_span_prompt",
        questions=questions,
        answers=answers,
        contexts=contexts_k4,
        prompt_mode="exact_span"
    )
)

results.append(
    run_prompt_experiment(
        name="best_guess_prompt",
        questions=questions,
        answers=answers,
        contexts=contexts_k4,
        prompt_mode="best_guess"
    )
)

In [ ]:
print("\n=== Prompt Experiment Results ===")
print(f"{'Method':<22} {'EM':<12} {'F1':<12}")
print("-" * 50)
for r in results:
    print(f"{r['name']:<22} {r['EM']:<12.6f} {r['F1']:<12.6f}")

In [ ]:
best_prompt_mode = "short"   # change if exact_span does better

contexts_k2 = build_contexts(questions, k=2)
contexts_k8 = build_contexts(questions, k=8)

retrieval_results = []

retrieval_results.append(
    run_prompt_experiment(
        name="retrieval_k2",
        questions=questions,
        answers=answers,
        contexts=contexts_k2,
        prompt_mode=best_prompt_mode
    )
)

retrieval_results.append(
    run_prompt_experiment(
        name="retrieval_k4",
        questions=questions,
        answers=answers,
        contexts=contexts_k4,
        prompt_mode=best_prompt_mode
    )
)

retrieval_results.append(
    run_prompt_experiment(
        name="retrieval_k8",
        questions=questions,
        answers=answers,
        contexts=contexts_k8,
        prompt_mode=best_prompt_mode
    )
)

In [ ]:
print("\n=== Retrieval Depth Results ===")
print(f"{'Method':<18} {'EM':<12} {'F1':<12}")
print("-" * 45)
for r in retrieval_results:
    print(f"{r['name']:<18} {r['EM']:<12.6f} {r['F1']:<12.6f}")

In [ ]:
prompt_names = [r["name"] for r in results]
prompt_ems = [r["EM"] for r in results]
prompt_f1s = [r["F1"] for r in results]

plt.figure(figsize=(8, 5))
plt.bar(range(len(prompt_names)), prompt_ems, label="EM", alpha=0.7)
plt.bar(range(len(prompt_names)), prompt_f1s, label="F1", alpha=0.7)
plt.xticks(range(len(prompt_names)), prompt_names, rotation=25, ha="right")
plt.ylabel("Score")
plt.title("Prompt Variant Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("final_results/prompt_variant_comparison.png", dpi=150)
plt.show()

In [ ]:
k_vals = [2, 4, 8]
retrieval_ems = [r["EM"] for r in retrieval_results]
retrieval_f1s = [r["F1"] for r in retrieval_results]

plt.figure(figsize=(8, 5))
plt.plot(k_vals, retrieval_ems, marker="o", label="EM")
plt.plot(k_vals, retrieval_f1s, marker="s", label="F1")
plt.xlabel("Number of retrieved passages (k)")
plt.ylabel("Score")
plt.title(f"Retrieval Depth with {best_prompt_mode} Prompt")
plt.xticks(k_vals)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("final_results/retrieval_depth_comparison.png", dpi=150)
plt.show()

In [ ]:
def show_examples(base_preds, improved_preds, answers, questions, n=10):
    shown = 0
    for q, a, bp, ip in zip(questions, answers, base_preds, improved_preds):
        if shown >= n:
            break
        if bp != ip:
            print("=" * 80)
            print("Q:", q)
            print("Gold:", a)
            print("Baseline:", bp)
            print("Improved:", ip)
            shown += 1

In [ ]:
baseline_preds = results[0]["predictions"]
short_preds = results[1]["predictions"]

show_examples(baseline_preds, short_preds, answers, questions, n=10)

In [ ]:
summary_path = "final_results/summary_results.txt"
with open(summary_path, "w") as f:
    f.write("Prompt Experiments\n")
    f.write("Method\tEM\tF1\n")
    for r in results:
        f.write(f"{r['name']}\t{r['EM']:.6f}\t{r['F1']:.6f}\n")

    f.write("\nRetrieval Depth Experiments\n")
    f.write("Method\tEM\tF1\n")
    for r in retrieval_results:
        f.write(f"{r['name']}\t{r['EM']:.6f}\t{r['F1']:.6f}\n")

print("Saved summary to", summary_path)

In [ ]:
exact_prompts = [build_prompt(q, ctx, "exact") for q, ctx in zip(questions, contexts)]
guess_prompts = [build_prompt(q, ctx, "best_guess") for q, ctx in zip(questions, contexts)]

In [ ]:
exact_preds = generate_batch(exact_prompts)
guess_preds = generate_batch(guess_prompts)

In [ ]:
exact_metrics = evaluate_predictions(exact_preds, answers)
guess_metrics = evaluate_predictions(guess_preds, answers)

print("\n=== Extra Prompt Results ===")
print("Exact-span:", exact_metrics)
print("Best-guess:", guess_metrics)

In [ ]:
Exact-span: {'EM': 0.0132, 'F1': 0.0711}
Best-guess: {'EM': 0.0095, 'F1': 0.0678}